In [ ]:
import pandas as pd
import glob
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
import statsmodels.formula.api as smf
from sklearn.model_selection import train_test_split
from sklearn import svm
from sklearn.metrics import precision_recall_fscore_support
import matplotlib.pyplot as plt


In [ ]:
csv_files = glob.glob("data/nba_shots/*.csv")

shots = pd.concat((pd.read_csv(file) for file in csv_files), ignore_index=True)
len(shots)

In [ ]:
# Looking at data
# print(shots.ZONE_NAME.unique())
# shots.ZONE_RANGE.unique()
# shots.SHOT_TYPE.unique()
# shots.EVENT_TYPE.unique()
# shots.ACTION_TYPE.unique()
# print(shots.LOC_X.describe())
# shots.LOC_Y.describe()
# shots.SHOT_DISTANCE.describe()
# shots.BASIC_ZONE.unique()
# shots.QUARTER.value_counts()

In [ ]:
# Predictors - for sure


# Predictors - maybe
# 'SHOT_TYPE', 'BASIC_ZONE', 'ZONE_NAME', 'ZONE_RANGE', 'LOC_X', 'LOC_Y', 'SHOT_DISTANCE', 'QUARTER', 
# 'MINS_LEFT', 'SECS_LEFT', 'ACTION_TYPE'

# Variables measuring the same thing in different ways
# - Distance
#   'SHOT_DISTANCE', 'ZONE_RANGE', 'LOC_X', 'LOC_Y', 'BASIC_ZONE', 'SHOT_TYPE'
#   - Can start with just distance

# - Location
#   'BASIC_ZONE', 'ZONE_NAME', 'LOC_X', 'LOC_Y'
#   - Could use angle
#   - And left/right/center

# - Time
#   'QUARTER', 'MINS_LEFT', 'SECS_LEFT'
#   - Interaction between quarter and seconds left in the quarter

# - Shot Type
#   'ACTION_TYPE', 'SHOT_TYPE'
#   - Want to use action type, it will just need some data cleaning (probably the next iteration of the model)

shots = shots[shots.QUARTER < 8]

shots['right'] = np.degrees(np.arctan2(shots.LOC_Y, shots.LOC_X)) <= 60
shots['left'] = np.degrees(np.arctan2(shots.LOC_Y, shots.LOC_X)) >= 120
shots['angle'] = abs(90 - np.degrees(np.arctan2(shots.LOC_Y, shots.LOC_X)))

shots['quarter_1'] = shots.QUARTER == 1
shots['quarter_2'] = shots.QUARTER == 2
shots['quarter_3'] = shots.QUARTER == 3
shots['quarter_4'] = shots.QUARTER == 4
shots['quarter_5'] = shots.QUARTER == 5
shots['quarter_6'] = shots.QUARTER == 6
shots['quarter_7'] = shots.QUARTER == 7
shots['total_secs_left'] = shots.MINS_LEFT * 60 + shots.SECS_LEFT
shots['non_jump_shot'] = ~shots.ACTION_TYPE.str.contains("Jump")

shots['shot_made'] = shots.SHOT_MADE.astype(int)


shots_X = shots[['SHOT_DISTANCE', 'right', 'left', 'angle', 'quarter_2', 'quarter_3', 'quarter_4',
                 'quarter_5', 'quarter_6', 'quarter_7', 'total_secs_left']]
shots_y = shots.SHOT_MADE.astype(int)

train_shots, test_shots = train_test_split(shots, test_size=0.05, random_state=42)

scaler = StandardScaler()

# Select numeric columns
numeric_cols = ['SHOT_DISTANCE', 'angle', 'total_secs_left']

# Fit and transform the numeric columns
train_shots[['shot_distance_std', 'angle_std', 'total_secs_left_std']] = scaler.fit_transform(train_shots[numeric_cols])
test_shots[['shot_distance_std', 'angle_std', 'total_secs_left_std']] = scaler.transform(test_shots[numeric_cols])

In [ ]:
len(test_shots)

In [ ]:
model = smf.ols(formula="shot_made ~ shot_distance_std + angle_std + quarter_2 + quarter_3 + quarter_4 + quarter_5 + quarter_6 + quarter_7 + total_secs_left_std", data=train_shots).fit()

# View the comprehensive R-style summary table
print(model.summary())

Quarter effect is interesting, distance and seconds make sense, I'm confused on the angle effect though.

In [ ]:
probs = model.predict(test_shots)
preds = probs > 0.5

precision_recall_fscore_support(test_shots.shot_made, preds)

In [ ]:
model = smf.ols(formula="shot_made ~ shot_distance_std + right + left + quarter_2 + quarter_3 + quarter_4 + quarter_5 + quarter_6 + quarter_7 + total_secs_left_std", data=train_shots).fit()

# View the comprehensive R-style summary table
print(model.summary())

In [ ]:
probs = model.predict(test_shots)
preds = probs > 0.5

precision_recall_fscore_support(test_shots.shot_made, preds)

In [ ]:
# shots.ACTION_TYPE.unique()

In [ ]:
# print(len(shots))
# print(len(shots[shots.ACTION_TYPE.str.contains("Jump")]))
# print(len(shots[shots.ACTION_TYPE.str.contains("Dunk")]))
# print(len(shots[shots.ACTION_TYPE.str.contains("Layup")]))
# print(len(shots[shots.ACTION_TYPE.str.contains("Hook")]))
# print(len(shots[shots.ACTION_TYPE.str.contains("Tip")]))
# print(len(shots[shots.ACTION_TYPE.str.contains("Finger Roll")]))

In [ ]:
model = smf.ols(formula="shot_made ~ non_jump_shot + shot_distance_std + right + left + quarter_2 + quarter_3 + quarter_4 + quarter_5 + quarter_6 + quarter_7 + total_secs_left_std + non_jump_shot:shot_distance_std + non_jump_shot:right + non_jump_shot:left + non_jump_shot:quarter_2 + non_jump_shot:quarter_3 + non_jump_shot:quarter_4 + non_jump_shot:quarter_5 + non_jump_shot:quarter_6 + non_jump_shot:quarter_7 + non_jump_shot:total_secs_left_std", data=train_shots).fit()

# View the comprehensive R-style summary table
print(model.summary())

In [ ]:
probs = model.predict(test_shots)

preds = probs > 0.4

precision, recall, fscore, _ = precision_recall_fscore_support(test_shots.shot_made, preds)

In [ ]:
class_names = np.unique(test_shots.shot_made)
class_names

In [ ]:
data = {
    'Metric': ['Precision', 'Recall', 'F1-Score'],
    'Class 0': [precision[0], recall[0], fscore[0]],
    'Class 1': [precision[1], recall[1], fscore[1]]
}
df = pd.DataFrame(data).set_index('Metric')

# 4. Plot side-by-side grouped bar chart
ax = df.plot(kind='bar', figsize=(8, 5), color=['#95a5a6', '#34495e'], width=0.6)

# Add text values on top of the bars
for p in ax.patches:
    ax.annotate(f"{p.get_height():.2f}", 
                (p.get_x() + p.get_width() / 2., p.get_height()), 
                ha='center', va='center', 
                xytext=(0, 8), 
                textcoords='offset points')

plt.title("Per-Class Classification Metrics Comparison")
plt.ylabel("Score")
plt.ylim(0, 1.1)
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.legend(title="Classes")
plt.show()

In [ ]:
# Attempted SVM
# shots_X = shots[['non_jump_shot', 'shot_distance_std', 'right', 'left', 'quarter_2', 'quarter_3', 'quarter_4',
#                  'quarter_5', 'quarter_6', 'quarter_7', 'total_secs_left_std',]]
# shots_y = shots.shot_made
#
# clf = svm.SVC(kernel='linear', C=1, class_weight={0:1, 1:1})
# clf = svm.SVC(kernel='poly', C=1, degree=1, class_weight={0:1, 1:.5}) # try degree=2,4
# clf = svm.SVC(kernel='rbf', C=1, gamma=0.8) # try gamma=.1,1
# clf.fit(shots_X.values, shots_y)